<a href="https://colab.research.google.com/github/Innej3/PatientZero/blob/main/PatientZero-SIR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
S, I_, R_ = 0, 1, 2

In [ ]:
'''

Added a thir State:
R - recovered - no longer infected, cant infect anyone cant, canth catch it again
Reocver prob - Gamma

If R = Beta/ Gamma > 1, the outbreak grows, below it dies out
'''

In [ ]:
def generate_topology(case='BA', n=90, seed=42):
  rng = np.random.default_rng(seed)
  if case == 'ER':
    G = nx.erdos_renyi_graph(n, p=0.065, seed=seed)
  elif case == 'BA':
    G = nx.barabasi_albert_graph(n, 2, seed=seed)
  elif case == 'WS':
    G = nx.watts_strogatz_graph(n, 4, 0.15, seed=seed)
  else:
    raise ValueError('Invalid TOPO')

  if not nx.is_connected(G):
    largest_cc = max(nx.connected_components(G), key=len)
    G = G.subgraph(largest_cc).copy()
    G = nx.convert_node_labels_to_integers(G)

  degrees = dict(G.degree())
  max_hub = max(degrees, key=degrees.get)


  if case == 'BA':
    lengths = nx.single_source_shortest_path_length(G, max_hub)
    candidates = [u for u,d in lengths.items() if d==2 and degrees[u] <= 2]
    patient_zero = candidates[0] if candidates else rng.choice(list(G.nodes()))
  else:
    patient_zero = rng.choice(list(G.nodes()))

  pos = nx.spring_layout(G, k=0.38, iterations=80, seed=seed)
  return G, pos, patient_zero, max_hub

In [ ]:
def simulate_sir(adj, n, source, beta, gamma, target_frac, rng, max_steps=200):
  """Synchronous SIR. Stops at the first step where ever-infected >= target.
     Returns the state array, or None if the outbreak dies out first."""
  state = np.zeros(n, dtype=int)
  state[source] = I_
  target = int(target_frac * n)
  for _ in range(max_steps):
    if (state > 0).sum() >= target:
      return state
    inf = np.flatnonzero(state == I_)
    if len(inf) == 0:
      return None
    new_inf = set()
    for u in inf:
      for v in adj[u]:
        if state[v] == S and rng.random() < beta:
          new_inf.add(v)
    recover = inf[rng.random(len(inf)) < gamma]     # only nodes infected before this step can recover
    if new_inf:
      state[list(new_inf)] = I_
    state[recover] = R_
  return state if (state > 0).sum() >= target else None


In [ ]:
def jaccard(a, b):
  if not a and not b:
    return 1.0
  return len(a & b) / len(a | b)

In [ ]:
def abc_sir(adj, n, obs_I, obs_R, mode, candidates, beta, gamma, target_frac,
            n_trials, rng, eps=0.12):
  scores = np.zeros(n)
  for c in candidates:
    acc = 0.0
    for _ in range(n_trials):
      st = simulate_sir(adj, n, c, beta, gamma, target_frac, rng)
      if st is None:
        continue                      # died out: can't produce the observation
      sI = set(np.flatnonzero(st == I_).tolist())
      sR = set(np.flatnonzero(st == R_).tolist())
      if mode == 'I':
        J = jaccard(sI, obs_I)
      elif mode == 'IR':
        J = jaccard(sI | sR, obs_I | obs_R)
      else:                           # 'labels'
        parts = [jaccard(sI, obs_I)]
        if obs_R or sR:
          parts.append(jaccard(sR, obs_R))
        J = np.mean(parts)
      acc += np.exp((J - 1.0) / eps)
    scores[c] = acc / n_trials
  post = np.zeros(n)
  if scores.sum() > 0:
    post = scores / scores.sum()
  else:
    post[list(candidates)] = 1.0 / len(candidates)
  return post


In [ ]:
def tie_rank(post, src):
  p = post[src]
  higher = int((post > p + 1e-12).sum())
  ties = int((np.abs(post - p) <= 1e-12).sum())
  return higher + (ties + 1) / 2

In [ ]:
def jordan_posterior(G, infected_set, tau=0.6):

  G_I = G.subgraph(infected_set)
  post = np.zeros(len(G))
  if len(infected_set) == 1:
    only_node = next(iter(infected_set))
    post[only_node] = 1.0
    return post

  ecc = {}
  for v in infected_set:
    lengths = nx.single_source_shortest_path_length(G_I, v)
    ecc[v] = max(lengths.values())

  min_ecc = min(ecc.values())
  logits = np.array([-(ecc[v] - min_ecc) / tau for v in infected_set])
  weights = np.exp(logits)
  weights /= weights.sum()

  for v, w in zip(infected_set, weights):
    post[v] = w
  return post

In [ ]:
def compute_rumor_posterior(G, infected_set, temp=2.5):
  """
  Shah-zaman Rumor Centrality - in how many different orders could the infection have
  spread if it started at v? the more possible ordering the more likely v is the source
  Rumor centrality assigns a score to every node in the infected subgraph.
  The node with the highest score is identified as the estimated source
  (the "rumor center").

  Since our network has cycles, we use build a BFS tree from each candidate root:
  keeping only the first edges that reaches each node.
  """
  G_I = G.subgraph(infected_set)
  post = np.zeros(len(G))
  if len(infected_set) == 1:
    only_node = next(iter(infected_set))
    post[only_node] = 1.0
    return post

  log_scores = {}
  for root in infected_set:
    bfs_edges = list(nx.bfs_edges(G_I, root))
    children = {u: [] for u in infected_set}
    order = [root]
    for u, v in bfs_edges:
      children[u].append(v)
      order.append(v)

    subtree_size = {u: 1 for u in infected_set}
    for u in reversed(order):
      for ch in children[u]:
        subtree_size[u] += subtree_size[ch]

    log_scores[root] = -sum(np.log(subtree_size[u]) / temp for u in infected_set)

  max_log = max(log_scores.values())
  weights = np.array([np.exp(log_scores[v] - max_log) for v in infected_set])
  weights /= weights.sum()

  for v, w in zip(infected_set, weights):
    post[v] = w
  return post


In [ ]:
def evaluate_sir(case='BA', gammas=(0.0, 0.1, 0.2, 0.3), modes=('I', 'IR', 'labels'),
                 n_runs=20, beta=0.3, frac=0.3, n_trials=20, tau=0.6):
  '''
  Three observation modes
  I - see only whos currently infected
  IR - see everyone ever infected, but cant tell infected from recovered
  labels - see everyone ever infected and whos recovered

  We uesd Jordan centrality only in I case, cause sick nodes may not be connected to
  each others, due to recovery. So we cant use the other two methods.
  '''

  for gamma in gammas:
    ranks = {}
    src_visible = []
    for r in range(n_runs):
      G, _, _, hub = generate_topology(case=case, n=85, seed=r)
      n = len(G)
      adj = [list(G.neighbors(u)) for u in range(n)]
      D = dict(nx.all_pairs_shortest_path_length(G))
      rng = np.random.default_rng(5000 + r)

      st = None
      for _ in range(20):
        src = int(rng.integers(n))
        st = simulate_sir(adj, n, src, beta, gamma, frac, rng)
        if st is not None:
          break
      if st is None:
        continue
      obs_I = set(np.flatnonzero(st == I_).tolist())
      obs_R = set(np.flatnonzero(st == R_).tolist())
      if not obs_I:
        continue
      IR = obs_I | obs_R
      src_visible.append(src in obs_I)

      for mode in modes:
        cand = list(range(n)) if mode == 'I' else sorted(IR)
        posts = {}
        if mode == 'I':
          ecc = np.array([max(D[v][u] for u in obs_I) for v in cand])
          w = np.exp(-(ecc - ecc.min()) / tau)
          p = np.zeros(n); p[cand] = w / w.sum()
          posts['jordan'] = p
        else:
          posts['jordan'] = jordan_posterior(G, IR)
          posts['rumor'] = compute_rumor_posterior(G, IR)
        posts['abc'] = abc_sir(adj, n, obs_I, obs_R, mode, cand, beta, gamma, frac, n_trials, rng)
        rnd = np.zeros(n); rnd[cand] = 1.0 / len(cand)
        posts['random'] = rnd
        for k, post in posts.items():
          ranks.setdefault((mode, k), []).append(tie_rank(post, src))

    print(f"\n{case}  gamma={gamma}  runs={len(src_visible)}  "
          f"source still infected (visible in I): {np.mean(src_visible):.2f}")
    for mode in modes:
      line = "  ".join(f"{k}: {np.median(ranks[(mode, k)]):5.1f}"
                       for k in ['random', 'jordan', 'rumor', 'abc'] if (mode, k) in ranks)
      print(f"  mode={mode:6s} median rank  {line}")

In [ ]:
evaluate_sir('BA')


BA  gamma=0.0  runs=20  source still infected (visible in I): 1.00
  mode=I      median rank  random:  43.0  jordan:  23.2  abc:   5.5
  mode=IR     median rank  random:  16.8  jordan:  14.0  rumor:  11.5  abc:   4.5
  mode=labels median rank  random:  16.8  jordan:  14.0  rumor:  11.5  abc:   6.0

BA  gamma=0.1  runs=20  source still infected (visible in I): 0.65
  mode=I      median rank  random:  43.0  jordan:  16.2  abc:  15.0
  mode=IR     median rank  random:  16.2  jordan:  13.0  rumor:  10.0  abc:   2.5
  mode=labels median rank  random:  16.2  jordan:  13.0  rumor:  10.0  abc:   5.5

BA  gamma=0.2  runs=20  source still infected (visible in I): 0.40
  mode=I      median rank  random:  43.0  jordan:  20.2  abc:  24.5
  mode=IR     median rank  random:  15.0  jordan:  12.8  rumor:  10.2  abc:   8.0
  mode=labels median rank  random:  15.0  jordan:  12.8  rumor:  10.2  abc:   5.0

BA  gamma=0.3  runs=20  source still infected (visible in I): 0.05
  mode=I      median rank  rando

In [ ]:
'''
We can see that finding the source becomes more and more difficult as recovery increases
if We obseve carefully in case where gamma = 0,
for ABC when mode = IR rank is 4.5 and when mode = labels rank if 6.0
That difference is enitrely from the random simulation draws

We can observe the same difference below in case of WS network

So ABC's rank can wobble by +- 1.5
'''

In [ ]:
evaluate_sir('WS')


WS  gamma=0.0  runs=20  source still infected (visible in I): 1.00
  mode=I      median rank  random:  43.0  jordan:   7.0  abc:   4.0
  mode=IR     median rank  random:  14.5  jordan:   3.8  rumor:   4.0  abc:   3.0
  mode=labels median rank  random:  14.5  jordan:   3.8  rumor:   4.0  abc:   3.5

WS  gamma=0.1  runs=20  source still infected (visible in I): 0.45
  mode=I      median rank  random:  43.0  jordan:  22.2  abc:  11.0
  mode=IR     median rank  random:  14.0  jordan:   6.8  rumor:   9.5  abc:   4.0
  mode=labels median rank  random:  14.0  jordan:   6.8  rumor:   9.5  abc:   3.0

WS  gamma=0.2  runs=20  source still infected (visible in I): 0.15
  mode=I      median rank  random:  43.0  jordan:  15.2  abc:   8.5
  mode=IR     median rank  random:  14.0  jordan:   7.0  rumor:   9.0  abc:   5.0
  mode=labels median rank  random:  14.0  jordan:   7.0  rumor:   9.0  abc:   3.0

WS  gamma=0.3  runs=20  source still infected (visible in I): 0.10
  mode=I      median rank  rando

In [ ]:
'''



''''